# 1. Import Libraries
นำเข้าไลบรารีที่จำเป็นสำหรับการวิเคราะห์ข้อมูล (EDA), Data Preprocessing, และการสร้างโมเดล Machine Learning


In [1]:
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import root_mean_squared_error, r2_score
import lightgbm as lgb

# ตั้งค่าสไตล์กราฟให้สวยงามและอ่านง่าย
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False


# 2. นำเข้าข้อมูล (Data Loading)
โหลดชุดข้อมูล Ames Housing Dataset ทั้งไฟล์ `train.csv` (มีตัวแปรเป้าหมาย `SalePrice`) และ `test.csv` สำหรับทำนายผล


In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"ขนาดชุดข้อมูล Train: {train.shape}")
print(f"ขนาดชุดข้อมูล Test:  {test.shape}")
train.head()


ขนาดชุดข้อมูล Train: (1460, 81)
ขนาดชุดข้อมูล Test:  (1459, 80)


# 3. สำรวจและทำความรู้จักลักษณะตัวแปร (Distinguish Attributes & Data Fields)

## Data fields ของ Ames Housing Dataset ทั้งหมด

### ทั้งหมด 80 Feature นะจ๊ะ

| Feature | คำอธิบาย (ภาษาไทย) |
| ----- | ----- |
| **SalePrice** | ราคาขายของอสังหาริมทรัพย์ในหน่วยดอลลาร์  **นี่คือตัวแปรเป้าหมายที่คุณต้องการจะทำนาย** |
| **MSSubClass** | ประเภทและระดับของอาคาร |
| **MSZoning** | การแบ่งผังเมืองหรือเขตพื้นที่ทั่วไป |
| **LotFrontage** | ความยาวของพื้นที่ดินที่ติดกับถนน (หน่วยฟุต) |
| **LotArea** | ขนาดของที่ดิน (หน่วยตารางฟุต) |
| **Street** | ประเภทของถนนทางเข้า |
| **Alley** | ประเภทของซอยทางเข้า |
| **LotShape** | รูปร่างทั่วไปของที่ดิน |
| **LandContour** | ความราบเรียบหรือระดับความสูงต่ำของที่ดิน |
| **Utilities** | ประเภทของสาธารณูปโภคที่มีให้ (เช่น น้ำ ไฟฟ้า) |
| **LotConfig** | รูปแบบการจัดสรรที่ดิน (เช่น ที่ดินหัวมุม, ที่ดินในซอยตัน) |
| **LandSlope** | ความลาดชันของที่ดิน |
| **Neighborhood** | ย่านหรือบริเวณที่ตั้งภายในขอบเขตเมือง Ames |
| **Condition1** | ความใกล้กับถนนสายหลักหรือทางรถไฟ |
| **Condition2** | ความใกล้กับถนนสายหลักหรือทางรถไฟ (ถ้ามีจุดที่สอง) |
| **BldgType** | ประเภทของที่อยู่อาศัย |
| **HouseStyle** | รูปแบบหรือสไตล์ของที่อยู่อาศัย |
| **OverallQual** | คุณภาพของวัสดุและการตกแต่งโดยรวม |
| **OverallCond** | คะแนนสภาพของตัวบ้านโดยรวม |
| **YearBuilt** | ปีที่ก่อสร้าง |
| **YearRemodAdd** | ปีที่รีโนเวทหรือต่อเติม |
| **RoofStyle** | รูปแบบของหลังคา |
| **RoofMatl** | วัสดุที่ใช้ทำหลังคา |
| **Exterior1st** | วัสดุที่ใช้ตกแต่งภายนอกบ้าน (หลัก) |
| **Exterior2nd** | วัสดุที่ใช้ตกแต่งภายนอกบ้าน (ถ้ามีมากกว่าหนึ่งชนิด) |
| **MasVnrType** | ประเภทของอิฐหรือหินประดับผนัง |
| **MasVnrArea** | พื้นที่ของอิฐหรือหินประดับผนัง (หน่วยตารางฟุต) |
| **ExterQual** | คุณภาพของวัสดุภายนอก |
| **ExterCond** | สภาพปัจจุบันของวัสดุภายนอก |
| **Foundation** | ประเภทของฐานราก |
| **BsmtQual** | ความสูงของชั้นใต้ดิน (ใช้ประเมินคุณภาพของพื้นที่ใต้ดิน) |
| **BsmtCond** | สภาพโดยรวมของชั้นใต้ดิน |
| **BsmtExposure** | การเข้าถึงหรือระดับของผนังชั้นใต้ดิน (เช่น มีทางเดินออกไปสวนได้) |
| **BsmtFinType1** | คุณภาพของพื้นที่ชั้นใต้ดินที่ตกแต่งแล้ว ส่วนที่ 1 |
| **BsmtFinSF1** | พื้นที่ชั้นใต้ดินที่ตกแต่งแล้ว ส่วนที่ 1 (หน่วยตารางฟุต) |
| **BsmtFinType2** | คุณภาพของพื้นที่ชั้นใต้ดินที่ตกแต่งแล้ว ส่วนที่ 2 (ถ้ามี) |
| **BsmtFinSF2** | พื้นที่ชั้นใต้ดินที่ตกแต่งแล้ว ส่วนที่ 2 (หน่วยตารางฟุต) |
| **BsmtUnfSF** | พื้นที่ชั้นใต้ดินที่ยังไม่ได้ตกแต่ง (หน่วยตารางฟุต) |
| **TotalBsmtSF** | พื้นที่ชั้นใต้ดินรวมทั้งหมด (หน่วยตารางฟุต) |
| **Heating** | ประเภทของระบบทำความร้อน |
| **HeatingQC** | คุณภาพและสภาพของระบบทำความร้อน |
| **CentralAir** | ระบบปรับอากาศแบบรวมศูนย์ (แอร์) |
| **Electrical** | ระบบไฟฟ้า |
| **1stFlrSF** | พื้นที่ใช้สอยชั้น 1 (หน่วยตารางฟุต) |
| **2ndFlrSF** | พื้นที่ใช้สอยชั้น 2 (หน่วยตารางฟุต) |
| **LowQualFinSF** | พื้นที่ที่ได้รับการตกแต่งแบบคุณภาพต่ำ (รวมทุกชั้น) (หน่วยตารางฟุต) |
| **GrLivArea** | พื้นที่ใช้สอยรวมเหนือระดับพื้นดิน (หน่วยตารางฟุต) |
| **BsmtFullBath** | จำนวนห้องน้ำเต็มรูปแบบในชั้นใต้ดิน |
| **BsmtHalfBath** | จำนวนห้องน้ำขนาดเล็กในชั้นใต้ดิน (ไม่มีที่อาบน้ำ) |
| **FullBath** | จำนวนห้องน้ำเต็มรูปแบบเหนือระดับพื้นดิน |
| **HalfBath** | จำนวนห้องน้ำขนาดเล็กเหนือระดับพื้นดิน |
| **Bedroom** | จำนวนห้องนอน (ไม่รวมชั้นใต้ดิน) |
| **Kitchen** | จำนวนห้องครัว |
| **KitchenQual** | คุณภาพของห้องครัว |
| **TotRmsAbvGrd** | จำนวนห้องทั้งหมดเหนือระดับพื้นดิน (ไม่รวมห้องน้ำ) |
| **Functional** | คะแนนระดับการใช้งานและความสมบูรณ์ของตัวบ้าน |
| **Fireplaces** | จำนวนเตาผิง |
| **FireplaceQu** | คุณภาพของเตาผิง |
| **GarageType** | ตำแหน่งหรือประเภทของโรงรถ |
| **GarageYrBlt** | ปีที่สร้างโรงรถ |
| **GarageFinish** | สภาพการตกแต่งภายในของโรงรถ |
| **GarageCars** | ขนาดของโรงรถ (นับจากจำนวนรถที่จอดได้) |
| **GarageArea** | ขนาดของโรงรถ (หน่วยตารางฟุต) |
| **GarageQual** | คุณภาพของโรงรถ |
| **GarageCond** | สภาพของโรงรถ |
| **PavedDrive** | ทางรถวิ่งที่ปูพื้นเรียบร้อยแล้ว |
| **WoodDeckSF** | พื้นที่ระเบียงไม้ (หน่วยตารางฟุต) |
| **OpenPorchSF** | พื้นที่ระเบียงหน้าบ้านแบบเปิด (หน่วยตารางฟุต) |
| **EnclosedPorch** | พื้นที่ระเบียงแบบปิดทึบ (หน่วยตารางฟุต) |
| **3SsnPorch** | พื้นที่ระเบียงกระจกสำหรับ 3 ฤดู (หน่วยตารางฟุต) |
| **ScreenPorch** | พื้นที่ระเบียงติดมุ้งลวด (หน่วยตารางฟุต) |
| **PoolArea** | พื้นที่สระว่ายน้ำ (หน่วยตารางฟุต) |
| **PoolQC** | คุณภาพของสระว่ายน้ำ |
| **Fence** | คุณภาพของรั้ว |
| **MiscFeature** | คุณลักษณะพิเศษอื่นๆ ที่ไม่ได้จัดอยู่ในหมวดใด |
| **MiscVal** | มูลค่าของคุณลักษณะพิเศษอื่นๆ (หน่วยดอลลาร์) |
| **MoSold** | เดือนที่ขายได้ |
| **YrSold** | ปีที่ขายได้ |
| **SaleType** | ประเภทของการขาย |
| **SaleCondition** | เงื่อนไขในการขาย |

# คำอธิบายตัวแปรชุดข้อมูล House Prices

> **หน่วยความยาว:** ฟุต (`ft`)  
> **หน่วยพื้นที่:** ตารางฟุต (`sq ft`)  
> **หน่วยมูลค่า:** ดอลลาร์สหรัฐ (`USD`)  
> **หมายเหตุ:** `NA` หมายถึง ไม่มีข้อมูล หรือไม่มีคุณลักษณะนั้น

## คำย่อระดับคุณภาพและสภาพ

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม (Excellent) |
| `Gd` | ดี (Good) |
| `TA` | มาตรฐาน/ค่าเฉลี่ย (Typical/Average) |
| `Fa` | พอใช้ (Fair) |
| `Po` | แย่ (Poor) |
| `NA` | ไม่มี/ไม่เกี่ยวข้อง (Not Available) |

---

### 3.1 รหัสระเบียน ประเภทบ้าน และที่ดิน

| Feature | คำอธิบาย |
|:---|:---|
| `Id` | รหัสประจำระเบียนของอสังหาริมทรัพย์ |
| `MSSubClass` | รหัสประเภทของที่อยู่อาศัยที่ขาย |
| `MSZoning` | ประเภทการแบ่งเขตการใช้ประโยชน์ที่ดิน |
| `LotFrontage` | ความยาวด้านหน้าที่ดินที่ติดถนน หน่วยเป็นฟุต |
| `LotArea` | ขนาดรวมของแปลงที่ดิน หน่วยเป็นตารางฟุต |
| `Street` | ประเภทถนนที่ใช้เข้าถึงอสังหาริมทรัพย์ |
| `Alley` | ประเภทซอยที่ใช้เข้าถึงอสังหาริมทรัพย์ |
| `LotShape` | รูปร่างของแปลงที่ดิน |
| `LandContour` | ลักษณะความราบหรือระดับของที่ดิน |
| `Utilities` | สาธารณูปโภคที่มีให้ใช้ |
| `LotConfig` | รูปแบบตำแหน่งของแปลงที่ดิน |
| `LandSlope` | ระดับความลาดชันของที่ดิน |
| `Neighborhood` | ย่านหรือทำเลทางกายภาพภายในเมือง Ames |
| `Condition1` | ความใกล้เคียงกับสภาพแวดล้อมภายนอกหลัก |
| `Condition2` | ความใกล้เคียงกับสภาพแวดล้อมภายนอกรอง |
| `BldgType` | ประเภทอาคารที่อยู่อาศัย |
| `HouseStyle` | รูปแบบหรือสไตล์ของบ้าน |

## `MSSubClass` — ประเภทที่อยู่อาศัย

| รหัส | ความหมาย |
|:---:|---|
| `20` | บ้าน 1 ชั้น สร้างปี 1946 หรือใหม่กว่า ทุกสไตล์ |
| `30` | บ้าน 1 ชั้น สร้างปี 1945 หรือเก่ากว่า |
| `40` | บ้าน 1 ชั้นที่มีห้องใต้หลังคาตกแต่งเสร็จแล้ว ทุกช่วงอายุ |
| `45` | บ้าน 1.5 ชั้น โดยพื้นที่ยังตกแต่งไม่เสร็จ ทุกช่วงอายุ |
| `50` | บ้าน 1.5 ชั้น ตกแต่งเสร็จแล้ว ทุกช่วงอายุ |
| `60` | บ้าน 2 ชั้น สร้างปี 1946 หรือใหม่กว่า |
| `70` | บ้าน 2 ชั้น สร้างปี 1945 หรือเก่ากว่า |
| `75` | บ้าน 2.5 ชั้น ทุกช่วงอายุ |
| `80` | บ้านแบบแบ่งระดับ หรือหลายระดับ |
| `85` | บ้านแบบ Split Foyer |
| `90` | บ้านแฝด ทุกสไตล์และทุกช่วงอายุ |
| `120` | บ้าน 1 ชั้นในโครงการ PUD สร้างปี 1946 หรือใหม่กว่า |
| `150` | บ้าน 1.5 ชั้นในโครงการ PUD ทุกช่วงอายุ |
| `160` | บ้าน 2 ชั้นในโครงการ PUD สร้างปี 1946 หรือใหม่กว่า |
| `180` | บ้าน PUD แบบหลายระดับ รวมถึง Split Level และ Split Foyer |
| `190` | บ้านสำหรับ 2 ครอบครัวที่ดัดแปลงจากอาคารเดิม ทุกสไตล์และทุกช่วงอายุ |

> `PUD` (*Planned Unit Development*) คือโครงการที่อยู่อาศัยที่ได้รับการวางผังและพัฒนาอย่างเป็นระบบ

## `MSZoning` — เขตการใช้ประโยชน์ที่ดิน

| รหัส | ความหมาย |
|:---:|---|
| `A` | พื้นที่เกษตรกรรม |
| `C` | พื้นที่พาณิชยกรรม |
| `FV` | พื้นที่อยู่อาศัยแบบ Floating Village |
| `I` | พื้นที่อุตสาหกรรม |
| `RH` | พื้นที่อยู่อาศัยความหนาแน่นสูง |
| `RL` | พื้นที่อยู่อาศัยความหนาแน่นต่ำ |
| `RP` | พื้นที่อยู่อาศัยแบบสวนสาธารณะ ความหนาแน่นต่ำ |
| `RM` | พื้นที่อยู่อาศัยความหนาแน่นปานกลาง |

## `Street` — ประเภทถนน

| รหัส | ความหมาย |
|:---:|---|
| `Grvl` | ถนนลูกรัง |
| `Pave` | ถนนลาดยาง |

## `Alley` — ประเภทซอย

| รหัส | ความหมาย |
|:---:|---|
| `Grvl` | ซอยลูกรัง |
| `Pave` | ซอยลาดยาง |
| `NA` | ไม่มีทางเข้าผ่านซอย |

## `LotShape` — รูปร่างแปลงที่ดิน

| รหัส | ความหมาย |
|:---:|---|
| `Reg` | รูปร่างปกติ |
| `IR1` | รูปร่างไม่สม่ำเสมอเล็กน้อย |
| `IR2` | รูปร่างไม่สม่ำเสมอปานกลาง |
| `IR3` | รูปร่างไม่สม่ำเสมอมาก |

## `LandContour` — ลักษณะภูมิประเทศ

| รหัส | ความหมาย |
|:---:|---|
| `Lvl` | พื้นที่เกือบราบหรือราบ |
| `Bnk` | ที่ดินยกสูงจากระดับถนนอย่างรวดเร็วและมีนัยสำคัญ |
| `HLS` | ที่ดินมีความลาดชันอย่างมีนัยสำคัญจากถนนไปยังตัวอาคาร |
| `Low` | พื้นที่ลุ่ม แอ่งต่ำ หรือต่ำกว่าระดับถนน |

## `Utilities` — สาธารณูปโภค

| รหัส | ความหมาย |
|:---:|---|
| `AllPub` | มีสาธารณูปโภคสาธารณะครบ ได้แก่ ไฟฟ้า ก๊าซ น้ำ และระบบท่อระบายน้ำ |
| `NoSewr` | มีไฟฟ้า ก๊าซ และน้ำ แต่ไม่มีระบบท่อระบายน้ำสาธารณะ โดยใช้ถังบำบัดน้ำเสีย |
| `NoSeWa` | มีเฉพาะไฟฟ้าและก๊าซ |
| `ELO` | มีเฉพาะไฟฟ้า |

## `LotConfig` — ตำแหน่งแปลงที่ดิน

| รหัส | ความหมาย |
|:---:|---|
| `Inside` | แปลงที่ดินด้านใน |
| `Corner` | แปลงหัวมุม |
| `CulDSac` | แปลงที่อยู่ปลายถนนตัน |
| `FR2` | มีหน้าติดถนน 2 ด้าน |
| `FR3` | มีหน้าติดถนน 3 ด้าน |

## `LandSlope` — ความลาดชัน

| รหัส | ความหมาย |
|:---:|---|
| `Gtl` | ลาดชันเล็กน้อย |
| `Mod` | ลาดชันปานกลาง |
| `Sev` | ลาดชันมาก |

## `Neighborhood` — ย่านที่ตั้ง

| รหัส | ความหมาย |
|:---:|---|
| `Blmngtn` | ย่าน Bloomington Heights |
| `Blueste` | ย่าน Bluestem |
| `BrDale` | ย่าน Briardale |
| `BrkSide` | ย่าน Brookside |
| `ClearCr` | ย่าน Clear Creek |
| `CollgCr` | ย่าน College Creek |
| `Crawfor` | ย่าน Crawford |
| `Edwards` | ย่าน Edwards |
| `Gilbert` | ย่าน Gilbert |
| `IDOTRR` | ย่าน Iowa DOT และแนวรถไฟ |
| `MeadowV` | ย่าน Meadow Village |
| `Mitchel` | ย่าน Mitchell |
| `Names` | ย่าน North Ames |
| `NoRidge` | ย่าน Northridge |
| `NPkVill` | ย่าน Northpark Villa |
| `NridgHt` | ย่าน Northridge Heights |
| `NWAmes` | ย่าน Northwest Ames |
| `OldTown` | ย่านเมืองเก่า |
| `SWISU` | ย่าน South & West of Iowa State University |
| `Sawyer` | ย่าน Sawyer |
| `SawyerW` | ย่าน Sawyer West |
| `Somerst` | ย่าน Somerset |
| `StoneBr` | ย่าน Stone Brook |
| `Timber` | ย่าน Timberland |
| `Veenker` | ย่าน Veenker |

## `Condition1` และ `Condition2` — สภาพแวดล้อมใกล้เคียง

| รหัส | ความหมาย |
|:---:|---|
| `Artery` | ติดถนนสายหลัก |
| `Feedr` | ติดถนนสายรอง |
| `Norm` | ทำเลปกติ |
| `RRNn` | อยู่ภายในระยะ 200 ฟุตจากทางรถไฟแนวเหนือ-ใต้ แต่ไม่ติดโดยตรง |
| `RRAn` | ติดทางรถไฟแนวเหนือ-ใต้ |
| `PosN` | อยู่ใกล้สิ่งแวดล้อมเชิงบวก เช่น สวนสาธารณะหรือพื้นที่สีเขียว |
| `PosA` | ติดกับสิ่งแวดล้อมเชิงบวก |
| `RRNe` | อยู่ภายในระยะ 200 ฟุตจากทางรถไฟแนวตะวันออก-ตะวันตก แต่ไม่ติดโดยตรง |
| `RRAe` | ติดทางรถไฟแนวตะวันออก-ตะวันตก |

## `BldgType` — ประเภทอาคาร

| รหัส | ความหมาย |
|:---:|---|
| `1Fam` | บ้านเดี่ยวสำหรับ 1 ครอบครัว |
| `2FmCon` | บ้าน 2 ครอบครัวที่ดัดแปลงจากบ้านเดี่ยว |
| `Duplx` | บ้านแฝด |
| `TwnhsE` | ทาวน์เฮาส์ยูนิตริม |
| `TwnhsI` | ทาวน์เฮาส์ยูนิตกลาง |

## `HouseStyle` — รูปแบบบ้าน

| รหัส | ความหมาย |
|:---:|---|
| `1Story` | บ้าน 1 ชั้น |
| `1.5Fin` | บ้าน 1.5 ชั้น โดยชั้นบนตกแต่งเสร็จแล้ว |
| `1.5Unf` | บ้าน 1.5 ชั้น โดยชั้นบนยังไม่ตกแต่ง |
| `2Story` | บ้าน 2 ชั้น |
| `2.5Fin` | บ้าน 2.5 ชั้น โดยชั้นบนตกแต่งเสร็จแล้ว |
| `2.5Unf` | บ้าน 2.5 ชั้น โดยชั้นบนยังไม่ตกแต่ง |
| `SFoyer` | บ้านแบบ Split Foyer |
| `SLvl` | บ้านแบบแบ่งระดับ |

---

### 3.2 คุณภาพ การก่อสร้าง และภายนอกบ้าน

| Feature | คำอธิบาย |
|:---|:---|
| `OverallQual` | คะแนนคุณภาพวัสดุและการตกแต่งโดยรวมของบ้าน |
| `OverallCond` | คะแนนสภาพโดยรวมของบ้าน |
| `YearBuilt` | ปีที่สร้างบ้านครั้งแรก |
| `YearRemodAdd` | ปีที่ปรับปรุงหรือต่อเติมล่าสุด |
| `RoofStyle` | รูปแบบหลังคา |
| `RoofMatl` | วัสดุมุงหลังคา |
| `Exterior1st` | วัสดุผนังภายนอกหลัก |
| `Exterior2nd` | วัสดุผนังภายนอกรอง |
| `MasVnrType` | ประเภทวัสดุตกแต่งผนังภายนอก |
| `MasVnrArea` | พื้นที่ของวัสดุตกแต่งผนังภายนอก หน่วยเป็นตารางฟุต |
| `ExterQual` | คุณภาพวัสดุภายนอกบ้าน |
| `ExterCond` | สภาพวัสดุภายนอกบ้าน |
| `Foundation` | ประเภทฐานราก |

## `OverallQual` และ `OverallCond`

| คะแนน | ความหมาย |
|:---:|---|
| `10` | ดีเยี่ยมมาก |
| `9` | ดีเยี่ยม |
| `8` | ดีมาก |
| `7` | ดี |
| `6` | สูงกว่าค่าเฉลี่ยเล็กน้อย |
| `5` | ค่าเฉลี่ย |
| `4` | ต่ำกว่าค่าเฉลี่ยเล็กน้อย |
| `3` | พอใช้ |
| `2` | แย่ |
| `1` | แย่มาก |

## `RoofStyle` — รูปแบบหลังคา

| รหัส | ความหมาย |
|:---:|---|
| `Flat` | หลังคาแบน |
| `Gable` | หลังคาจั่ว |
| `Gambrel` | หลังคาทรงโรงนา |
| `Hip` | หลังคาปั้นหยา |
| `Mansard` | หลังคาแมนซาร์ด |
| `Shed` | หลังคาเพิงหมาแหงน |

## `RoofMatl` — วัสดุหลังคา

| รหัส | ความหมาย |
|:---:|---|
| `ClyTile` | กระเบื้องดินเผา |
| `CompShg` | กระเบื้องแอสฟัลต์คอมโพสิต |
| `Membran` | แผ่นเมมเบรน |
| `Metal` | โลหะ |
| `Roll` | วัสดุมุงหลังคาแบบม้วน |
| `Tar&Grv` | ยางมะตอยและกรวด |
| `WdShake` | แผ่นไม้แบบ Shake |
| `WdShngl` | กระเบื้องไม้ |

## `Exterior1st` และ `Exterior2nd` — วัสดุผนังภายนอก

| รหัส | ความหมาย |
|:---:|---|
| `AsbShng` | แผ่นกระเบื้องใยหิน |
| `AsphShn` | แผ่นกระเบื้องแอสฟัลต์ |
| `BrkComm` | อิฐทั่วไป |
| `BrkFace` | อิฐโชว์หน้า |
| `CBlock` | คอนกรีตบล็อก |
| `CemntBd` | แผ่นซีเมนต์ |
| `HdBoard` | ฮาร์ดบอร์ด |
| `ImStucc` | ปูนปั้นเลียนแบบ |
| `MetalSd` | ผนังโลหะ |
| `Other` | วัสดุอื่น |
| `Plywood` | ไม้อัด |
| `PreCast` | คอนกรีตสำเร็จรูป |
| `Stone` | หิน |
| `Stucco` | ปูนปั้น |
| `VinylSd` | ผนังไวนิล |
| `Wd Sdng` | ผนังไม้ |
| `WdShing` | แผ่นไม้ชิงเกิล |

## `MasVnrType` — วัสดุตกแต่งผนัง

| รหัส | ความหมาย |
|:---:|---|
| `BrkCmn` | อิฐทั่วไป |
| `BrkFace` | อิฐโชว์หน้า |
| `CBlock` | คอนกรีตบล็อก |
| `None` | ไม่มีวัสดุตกแต่ง |
| `Stone` | หิน |

## `ExterQual` และ `ExterCond`

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `Po` | แย่ |

## `Foundation` — ฐานราก

| รหัส | ความหมาย |
|:---:|---|
| `BrkTil` | ฐานรากอิฐและกระเบื้อง |
| `CBlock` | ฐานรากคอนกรีตบล็อก |
| `PConc` | ฐานรากคอนกรีตหล่อ |
| `Slab` | ฐานรากแบบแผ่นพื้นคอนกรีต |
| `Stone` | ฐานรากหิน |
| `Wood` | ฐานรากไม้ |

---

### 3.3 ห้องใต้ดินและระบบภายในบ้าน

| Feature | คำอธิบาย |
|:---|:---|
| `BsmtQual` | คุณภาพหรือความสูงของห้องใต้ดิน |
| `BsmtCond` | สภาพทั่วไปของห้องใต้ดิน |
| `BsmtExposure` | ระดับการเปิดโล่งของห้องใต้ดิน |
| `BsmtFinType1` | ประเภทพื้นที่ตกแต่งเสร็จในห้องใต้ดินส่วนที่ 1 |
| `BsmtFinSF1` | พื้นที่ห้องใต้ดินที่ตกแต่งเสร็จส่วนที่ 1 |
| `BsmtFinType2` | ประเภทพื้นที่ตกแต่งเสร็จในห้องใต้ดินส่วนที่ 2 |
| `BsmtFinSF2` | พื้นที่ห้องใต้ดินที่ตกแต่งเสร็จส่วนที่ 2 |
| `BsmtUnfSF` | พื้นที่ห้องใต้ดินที่ยังไม่ตกแต่ง |
| `TotalBsmtSF` | พื้นที่รวมของห้องใต้ดิน |
| `Heating` | ประเภทระบบทำความร้อน |
| `HeatingQC` | คุณภาพและสภาพของระบบทำความร้อน |
| `CentralAir` | การมีระบบปรับอากาศส่วนกลาง |
| `Electrical` | ประเภทระบบไฟฟ้า |

## `BsmtQual` — คุณภาพ/ความสูงห้องใต้ดิน

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม: ความสูงอย่างน้อย 100 นิ้ว |
| `Gd` | ดี: ความสูง 90–99 นิ้ว |
| `TA` | มาตรฐาน: ความสูง 80–89 นิ้ว |
| `Fa` | พอใช้: ความสูง 70–79 นิ้ว |
| `Po` | แย่: ความสูงน้อยกว่า 70 นิ้ว |
| `NA` | ไม่มีห้องใต้ดิน |

## `BsmtCond` — สภาพห้องใต้ดิน

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `Po` | แย่ |
| `NA` | ไม่มีห้องใต้ดิน |

## `BsmtExposure` — การเปิดโล่งของห้องใต้ดิน

| รหัส | ความหมาย |
|:---:|---|
| `Gd` | เปิดโล่งดี |
| `Av` | เปิดโล่งระดับเฉลี่ย |
| `Mn` | เปิดโล่งเล็กน้อย |
| `No` | ไม่มีการเปิดโล่ง |
| `NA` | ไม่มีห้องใต้ดิน |

## `BsmtFinType1` และ `BsmtFinType2`

| รหัส | ความหมาย |
|:---:|---|
| `GLQ` | พื้นที่อยู่อาศัยคุณภาพดี |
| `ALQ` | พื้นที่อยู่อาศัยคุณภาพเฉลี่ย |
| `BLQ` | พื้นที่อยู่อาศัยคุณภาพต่ำกว่าค่าเฉลี่ย |
| `Rec` | ห้องพักผ่อนหรือห้องสันทนาการ |
| `LwQ` | พื้นที่คุณภาพต่ำ |
| `Unf` | ยังไม่ตกแต่ง |
| `NA` | ไม่มีห้องใต้ดิน |

## `Heating` — ระบบทำความร้อน

| รหัส | ความหมาย |
|:---:|---|
| `Floor` | ระบบทำความร้อนใต้พื้น |
| `GasA` | ระบบลมร้อนใช้ก๊าซ |
| `GasW` | ระบบน้ำร้อนใช้ก๊าซ |
| `Grav` | ระบบทำความร้อนแบบแรงโน้มถ่วง |
| `OthW` | ระบบน้ำร้อนประเภทอื่น |
| `Wall` | เครื่องทำความร้อนติดผนัง |

## `HeatingQC` — คุณภาพและสภาพระบบทำความร้อน

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `Po` | แย่ |

## `CentralAir` — เครื่องปรับอากาศส่วนกลาง

| รหัส | ความหมาย |
|:---:|---|
| `Y` | มี |
| `N` | ไม่มี |

## `Electrical` — ระบบไฟฟ้า

| รหัส | ความหมาย |
|:---:|---|
| `SBrkr` | ระบบเบรกเกอร์มาตรฐาน |
| `FuseA` | กล่องฟิวส์ 60 แอมป์ และส่วนใหญ่ใช้สายไฟ Romex |
| `FuseF` | กล่องฟิวส์ 60 แอมป์ และส่วนใหญ่ใช้ระบบสายไฟแบบ Knob-and-Tube |
| `FuseP` | ฟิวส์แบบปลั๊ก 60 แอมป์ และส่วนใหญ่ใช้ระบบสายไฟแบบ Knob-and-Tube |
| `Mix` | ระบบไฟฟ้าแบบผสม |

---

### 3.4 พื้นที่ใช้สอยและห้องต่าง ๆ

| Feature | คำอธิบาย |
|:---|:---|
| `1stFlrSF` | พื้นที่ใช้สอยของชั้น 1 |
| `2ndFlrSF` | พื้นที่ใช้สอยของชั้น 2 |
| `LowQualFinSF` | พื้นที่ตกแต่งเสร็จคุณภาพต่ำของทุกชั้นรวมกัน |
| `GrLivArea` | พื้นที่อยู่อาศัยเหนือระดับพื้นดิน |
| `BsmtFullBath` | จำนวนห้องน้ำเต็มรูปแบบในห้องใต้ดิน |
| `BsmtHalfBath` | จำนวนห้องน้ำครึ่งรูปแบบในห้องใต้ดิน |
| `FullBath` | จำนวนห้องน้ำเต็มรูปแบบเหนือระดับพื้นดิน |
| `HalfBath` | จำนวนห้องน้ำครึ่งรูปแบบเหนือระดับพื้นดิน |
| `BedroomAbvGr` | จำนวนห้องนอนเหนือระดับพื้นดิน ไม่รวมห้องนอนในห้องใต้ดิน |
| `KitchenAbvGr` | จำนวนห้องครัวเหนือระดับพื้นดิน |
| `KitchenQual` | คุณภาพห้องครัว |
| `TotRmsAbvGrd` | จำนวนห้องทั้งหมดเหนือระดับพื้นดิน ไม่รวมห้องน้ำ |
| `Functional` | ระดับความสมบูรณ์ในการใช้งานของบ้าน |
| `Fireplaces` | จำนวนเตาผิง |
| `FireplaceQu` | คุณภาพเตาผิง |

> ในบางไฟล์ ตัวแปร `BedroomAbvGr` และ `KitchenAbvGr` อาจถูกย่อเป็น `Bedroom` และ `Kitchen`

## `KitchenQual` — คุณภาพห้องครัว

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `Po` | แย่ |

## `Functional` — ความสมบูรณ์ในการใช้งาน

| รหัส | ความหมาย |
|:---:|---|
| `Typ` | ใช้งานได้ปกติ |
| `Min1` | มีข้อบกพร่องเล็กน้อย ระดับ 1 |
| `Min2` | มีข้อบกพร่องเล็กน้อย ระดับ 2 |
| `Mod` | มีข้อบกพร่องปานกลาง |
| `Maj1` | มีข้อบกพร่องรุนแรง ระดับ 1 |
| `Maj2` | มีข้อบกพร่องรุนแรง ระดับ 2 |
| `Sev` | เสียหายรุนแรง |
| `Sal` | ใช้งานได้จำกัดมาก/เหมาะสำหรับกู้ซากเท่านั้น |

## `FireplaceQu` — คุณภาพเตาผิง

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม: เตาผิงก่ออิฐคุณภาพสูง |
| `Gd` | ดี: เตาผิงก่ออิฐทั่วไป |
| `TA` | มาตรฐาน: เตาผิงสำเร็จรูป |
| `Fa` | พอใช้: เตาผิงสำเร็จรูปคุณภาพต่ำ |
| `Po` | แย่: เตาผิงก่ออิฐที่มีสภาพไม่ดี |
| `NA` | ไม่มีเตาผิง |

---

### 3.5 โรงจอดรถและพื้นที่ภายนอก

| Feature | คำอธิบาย |
|:---|:---|
| `GarageType` | ตำแหน่งหรือประเภทโรงจอดรถ |
| `GarageYrBlt` | ปีที่สร้างโรงจอดรถ |
| `GarageFinish` | ระดับการตกแต่งภายในโรงจอดรถ |
| `GarageCars` | ความจุของโรงจอดรถ หน่วยเป็นจำนวนรถ |
| `GarageArea` | พื้นที่โรงจอดรถ |
| `GarageQual` | คุณภาพโรงจอดรถ |
| `GarageCond` | สภาพโรงจอดรถ |
| `PavedDrive` | ระดับการปูพื้นทางเข้ารถ |
| `WoodDeckSF` | พื้นที่ระเบียงไม้ |
| `OpenPorchSF` | พื้นที่เฉลียงเปิด |
| `EnclosedPorch` | พื้นที่เฉลียงปิด |
| `3SsnPorch` | พื้นที่เฉลียงสำหรับใช้งาน 3 ฤดู |
| `ScreenPorch` | พื้นที่เฉลียงที่มีมุ้งลวด |
| `PoolArea` | พื้นที่สระว่ายน้ำ |
| `PoolQC` | คุณภาพสระว่ายน้ำ |
| `Fence` | คุณภาพรั้ว |
| `MiscFeature` | สิ่งอำนวยความสะดวกพิเศษอื่น |
| `MiscVal` | มูลค่าของสิ่งอำนวยความสะดวกพิเศษอื่น |

## `GarageType` — ประเภทโรงจอดรถ

| รหัส | ความหมาย |
|:---:|---|
| `2Types` | มีโรงจอดรถมากกว่า 1 ประเภท |
| `Attchd` | โรงจอดรถติดกับตัวบ้าน |
| `Basment` | โรงจอดรถชั้นใต้ดิน |
| `BuiltIn` | โรงจอดรถที่เป็นส่วนหนึ่งของตัวบ้าน ซึ่งมักมีห้องอยู่เหนือโรงจอดรถ |
| `CarPort` | ที่จอดรถมีหลังคาแต่ไม่มีผนัง |
| `Detchd` | โรงจอดรถแยกจากตัวบ้าน |
| `NA` | ไม่มีโรงจอดรถ |

> รหัส `Basment` สะกดตามข้อมูลต้นฉบับ แม้คำภาษาอังกฤษมาตรฐานคือ `Basement`

## `GarageFinish` — ระดับการตกแต่งโรงจอดรถ

| รหัส | ความหมาย |
|:---:|---|
| `Fin` | ตกแต่งเสร็จสมบูรณ์ |
| `RFn` | ตกแต่งเสร็จแบบหยาบ/บางส่วน |
| `Unf` | ยังไม่ตกแต่ง |
| `NA` | ไม่มีโรงจอดรถ |

## `GarageQual` และ `GarageCond`

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `Po` | แย่ |
| `NA` | ไม่มีโรงจอดรถ |

## `PavedDrive` — ทางเข้ารถ

| รหัส | ความหมาย |
|:---:|---|
| `Y` | ปูพื้นทั้งหมด |
| `P` | ปูพื้นบางส่วน |
| `N` | ไม่ได้ปูพื้น |

## `PoolQC` — คุณภาพสระว่ายน้ำ

| รหัส | ความหมาย |
|:---:|---|
| `Ex` | ดีเยี่ยม |
| `Gd` | ดี |
| `TA` | มาตรฐาน/ค่าเฉลี่ย |
| `Fa` | พอใช้ |
| `NA` | ไม่มีสระว่ายน้ำ |

## `Fence` — คุณภาพรั้ว

| รหัส | ความหมาย |
|:---:|---|
| `GdPrv` | รั้วเพื่อความเป็นส่วนตัวคุณภาพดี |
| `MnPrv` | รั้วเพื่อความเป็นส่วนตัวคุณภาพปานกลาง |
| `GdWo` | รั้วไม้คุณภาพดี |
| `MnWw` | รั้วไม้/ลวดคุณภาพปานกลาง |
| `NA` | ไม่มีรั้ว |

## `MiscFeature` — สิ่งอำนวยความสะดวกพิเศษ

| รหัส | ความหมาย |
|:---:|---|
| `Elev` | ลิฟต์ |
| `Gar2` | โรงจอดรถหลังที่ 2 |
| `Othr` | สิ่งอำนวยความสะดวกอื่น |
| `Shed` | โรงเก็บของ |
| `TenC` | สนามเทนนิส |
| `NA` | ไม่มีสิ่งอำนวยความสะดวกพิเศษ |

---

### 3.6 ข้อมูลการขายและตัวแปรเป้าหมาย

| Feature | คำอธิบาย |
|:---|:---|
| `MoSold` | เดือนที่ขายบ้าน |
| `YrSold` | ปีที่ขายบ้าน |
| `SaleType` | ประเภทการขาย |
| `SaleCondition` | เงื่อนไขการขาย |
| `SalePrice` | ราคาขายของอสังหาริมทรัพย์ หน่วยเป็นดอลลาร์สหรัฐ — เป็นตัวแปรเป้าหมายสำหรับการพยากรณ์ราคา |

## `SaleType` — ประเภทการขาย

| รหัส | ความหมาย |
|:---:|---|
| `WD` | การขายทั่วไปผ่าน Warranty Deed |
| `CWD` | การขายผ่าน Warranty Deed ด้วยเงินสด |
| `VWD` | การขายผ่าน Warranty Deed ด้วยสินเชื่อทหารผ่านศึก |
| `New` | บ้านสร้างใหม่และขายเป็นครั้งแรก |
| `COD` | การขายตามคำสั่งศาลหรือผู้จัดการมรดก |
| `Con` | การขายแบบสัญญาผ่อนชำระ เงื่อนไขปกติ |
| `ConLw` | การขายแบบสัญญา โดยมีเงินดาวน์และดอกเบี้ยต่ำ |
| `ConLI` | การขายแบบสัญญา โดยมีอัตราดอกเบี้ยต่ำ |
| `ConLD` | การขายแบบสัญญา โดยมีเงินดาวน์ต่ำ |
| `Oth` | การขายประเภทอื่น |

## `SaleCondition` — เงื่อนไขการขาย

| รหัส | ความหมาย |
|:---:|---|
| `Normal` | การขายปกติ |
| `Abnorml` | การขายที่มีเงื่อนไขผิดปกติ เช่น การยึดทรัพย์ การขายทอดตลาด หรือการขายระยะสั้น |
| `AdjLand` | การขายที่ดินเปล่าซึ่งติดกับทรัพย์สินเดิม |
| `Alloca` | การจัดสรรหรือการขายทรัพย์สินร่วมที่มีเอกสารสิทธิ์แยกกัน |
| `Family` | การขายระหว่างสมาชิกครอบครัว |
| `Partial` | บ้านยังสร้างไม่เสร็จในวันที่ประเมินล่าสุด |

---

### 3.7 สรุปจำนวนตัวแปร

| กลุ่ม | จำนวน |
|:---|---:|
| รหัสระเบียน | 1 |
| ตัวแปรอธิบายอสังหาริมทรัพย์ | 79 |
| ตัวแปรเป้าหมาย (`SalePrice`) | 1 |
| **รวมทั้งหมด** | **81** |


# 4. วิเคราะห์ตัวแปรเป้าหมาย (Target Variable Analysis: SalePrice)
ทำการสำรวจการกระจายตัวของราคาบ้าน `SalePrice` เพื่อตรวจสอบการเบ้ (Skewness) และ Outliers เบื้องต้น


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# กราฟที่ 1: Histogram + KDE
sns.histplot(train['SalePrice'], kde=True, color='#2980B9', bins=40, ax=axes[0])
axes[0].set_title('Distribution of SalePrice (Original)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sale Price ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

# กราฟที่ 2: Boxplot
sns.boxplot(x=train['SalePrice'], color='#AED6F1', ax=axes[1])
axes[1].set_title('Boxplot of SalePrice', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sale Price ($)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Skewness ของ SalePrice: {train['SalePrice'].skew():.2f}")
print(f"Kurtosis ของ SalePrice: {train['SalePrice'].kurt():.2f}")


Skewness ของ SalePrice: 1.88
Kurtosis ของ SalePrice: 6.54


![Distribution Analysis](assets/distribution_analysis.jpg)


## การทำ Log Transformation บนตัวแปรเป้าหมาย
เนื่องจาก `SalePrice` มีการกระจายตัวที่เบ้ขวา (Right-skewed) ชัดเจน การใช้ $\log(1+x)$ หรือ `np.log1p()` จะช่วยปรับข้อมูลให้ใกล้เคียง Normal Distribution ซึ่งช่วยให้โมเดล Linear และ Tree-based ทำงานได้มีประสิทธิภาพยิ่งขึ้น อีกทั้งสอดคล้องกับเกณฑ์การวัดผล RMSLE


In [4]:
y_train = train['SalePrice']
y_train_log = np.log1p(y_train)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(y_train_log, kde=True, color='#27AE60', bins=40, ax=axes[0])
axes[0].set_title('Distribution of Log(SalePrice + 1)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Log(Sale Price)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

sns.boxplot(x=y_train_log, color='#A9DFBF', ax=axes[1])
axes[1].set_title('Boxplot of Log(SalePrice + 1)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Sale Price)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Skewness หลังทำ Log: {y_train_log.skew():.2f}")


Skewness หลังทำ Log: 0.12


# 5. การตรวจสอบและจัดการ Outliers (Outlier Detection & Removal)
ในชุดข้อมูล Ames Housing ผู้สร้างชุดข้อมูล (**Dean De Cock**) ได้ระบุไว้เป็นกรณีพิเศษว่า มีบ้านที่มีพื้นที่ใช้สอยเหนือดิน `GrLivArea > 4000` sq ft ที่เป็น Outlier ผิดปกติอย่างรุนแรง (Partial/Agricultural sales) ที่ขายในราคาต่ำกว่าปกติ ซึ่งควรตัดออก 2 หลัง เพื่อไม่ให้บิดเบือนโมเดล Regression


In [5]:
# ตรวจสอบความสัมพันธ์ GrLivArea vs SalePrice ก่อนตัด Outliers
plt.figure(figsize=(9, 5))
sns.scatterplot(data=train, x='GrLivArea', y='SalePrice', alpha=0.7, color='#2C3E50')
plt.axvline(x=4000, color='red', linestyle='--', label='GrLivArea = 4000 sq ft')
plt.title('GrLivArea vs SalePrice (Identifying Outliers)', fontsize=14, fontweight='bold')
plt.xlabel('Above Ground Living Area (sq ft)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

# ค้นหาบ้านที่ GrLivArea > 4000 และ SalePrice < 300,000 ตามคำแนะนำของ Dean De Cock
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
print(f"จำนวน Outliers ที่ผิดปกติอย่างชัดเจน: {len(outliers)} หลัง")
print(outliers[['Id', 'GrLivArea', 'SalePrice', 'SaleCondition']])

# ทำการลบ Outliers 2 หลังนี้ออกจากชุดข้อมูล Train
train = train.drop(index=outliers.index).reset_index(drop=True)
y_train = train['SalePrice']
y_train_log = np.log1p(y_train)

print(f"ขนาดชุดข้อมูล Train หลังตัด Outliers: {train.shape}")


จำนวน Outliers ที่ผิดปกติอย่างชัดเจน: 2 หลัง
        Id  GrLivArea  SalePrice SaleCondition
523    524       4676     184750       Partial
1298  1299       5642     160000       Partial
ขนาดชุดข้อมูล Train หลังตัด Outliers: (1458, 81)


# 6. รวมชุดข้อมูล Train และ Test เพื่อเตรียม Data Preprocessing (Data Integration)
นำ `train` (ตัด `SalePrice` ออก) มารวมกับ `test` เป็น `df` ชั่วคราว เพื่อให้ขั้นตอนการจัดการ Missing Values และ Feature Engineering ดำเนินการไปในแนวทางเดียวกันทั้งสองชุด

> [!NOTE]
> **หมายเหตุเชิงวิชาการเรื่อง Data Leakage:**
> การนำ `train` และ `test` มารวมกันเป็น `df` ก่อนทำการ Impute (โดยเฉพาะ `LotFrontage` ที่ใช้ค่ามัธยฐานแยกตาม Neighborhood) ถือเป็น Data Leakage ในทางทฤษฎีเล็กน้อย อย่างไรก็ตาม นี่เป็นธรรมเนียมปฏิบัติที่ยอมรับได้ในการแข่งขัน Kaggle เนื่องจากชุดข้อมูล Test ถูกมอบให้ล่วงหน้าทั้งหมดแล้ว ทั้งนี้เราตระหนัก รับรู้ และยอมรับข้อจำกัดนี้อย่างชัดเจน


In [6]:
df = pd.concat([train.drop(columns=['SalePrice']), test], axis=0, ignore_index=True)
print(f"ขนาดชุดข้อมูลรวม df: {df.shape}")
print(f"แถว 0 ถึง {len(train)-1} คือ Train, แถว {len(train)} ถึง {len(df)-1} คือ Test")


ขนาดชุดข้อมูลรวม df: (2917, 80)
แถว 0 ถึง 1457 คือ Train, แถว 1458 ถึง 2916 คือ Test


# 7. ตรวจสอบและจัดการค่าที่ขาดหาย (Handling Missing Values)
ในชุดข้อมูลนี้ ค่า `NA` ในหลายตัวแปรไม่ได้แปลว่าข้อมูลสูญหาย (Missing) แต่หมายถึง **"ไม่มีสิ่งนั้นอยู่จริงในบ้าน"** (Not Applicable)
เราจึงต้องจัดการตามบริบททางกายภาพของบ้านอย่างถูกต้อง


In [7]:
# ค้นหาคอลัมน์ที่มี Missing Values
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
missing_percent = (missing_data / len(df)) * 100
missing_table = pd.concat([missing_data, missing_percent], axis=1, keys=['Missing Count', 'Percent (%)'])

print(f"จำนวนคอลัมน์ที่มีค่าว่างทั้งหมด: {len(missing_table)} คอลัมน์")
print(missing_table.head(15))


จำนวนคอลัมน์ที่มีค่าว่างทั้งหมด: 34 คอลัมน์
              Missing Count  Percent (%)
PoolQC                 2908    99.691464
MiscFeature            2812    96.400411
Alley                  2719    93.212204
Fence                  2346    80.425094
MasVnrType             1766    60.541652
FireplaceQu            1420    48.680151
LotFrontage             486    16.660953
GarageQual              159     5.450806
GarageYrBlt             159     5.450806
GarageCond              159     5.450806
GarageFinish            159     5.450806
GarageType              157     5.382242
BsmtExposure             82     2.811107
BsmtCond                 82     2.811107
BsmtQual                 81     2.776826


### การแทนค่าที่ว่าง (Data Imputation) ตามลักษณะตัวแปร
1. **กลุ่ม Categorical ที่ 'NA' แปลว่า 'ไม่มีสิ่งอำนวยความสะดวกนั้น':** แทนค่าด้วย `'None'`
2. **กลุ่ม Numerical ที่สัมพันธ์กับการไม่มีสิ่งนั้น:** แทนค่าด้วย `0` (เช่น ไม่มีโรงรถ พื้นที่โรงรถก็เป็น 0)
3. **กลุ่มปีที่สร้างโรงรถ (`GarageYrBlt`):** หากไม่มีโรงรถ ให้แทนด้วยปีที่สร้างบ้าน (`YearBuilt`) เพื่อป้องกันปัญหาปี 0 ซึ่งเป็น Outlier รุนแรง
4. **กลุ่ม `LotFrontage` (ความยาวหน้าที่ดิน):** แทนด้วยค่ามัธยฐาน (Median) แยกตามรายย่าน (`Neighborhood`)
5. **กลุ่มที่เหลือ:** แทนค่าเชิงตัวเลขด้วย Median และค่าเชิงกลุ่มด้วย Mode (รองรับทั้ง Pandas 2 และ 3 ด้วย `is_numeric_dtype`)


In [8]:
# 1. กลุ่มที่ 'NA' แปลว่า 'ไม่มีสิ่งนั้น' (Categorical) -> เติมด้วย 'None'
cols_to_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType'
]
for col in cols_to_none:
    df[col] = df[col].fillna('None')

# 2. กลุ่มตัวเลขที่สัมพันธ์กับการไม่มีสิ่งนั้น -> เติมด้วย 0
cols_to_zero = [
    'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]
for col in cols_to_zero:
    df[col] = df[col].fillna(0)

# 3. จัดการ GarageYrBlt: ถ้าไม่มีโรงรถ ให้ใช้ปีที่สร้างบ้านแทน (เลี่ยงปัญหาปี ค.ศ. 0)
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

# 4. LotFrontage: เติมด้วยมัธยฐานของแต่ละย่าน (Neighborhood)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
# หากยังมีย่านใดที่ว่าง ให้เติมด้วยมัธยฐานรวม
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

# 5. Electrical ขาดเล็กน้อย -> เติมด้วย Mode
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# 6. คอลัมน์ที่เหลือที่ยังมีค่าว่าง (ใช้ is_numeric_dtype เพื่อความเสถียรบน Pandas ทุกเวอร์ชัน)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode_val = df[col].mode()
            df[col] = df[col].fillna(mode_val.iloc[0] if not mode_val.empty else 'None')

print("จำนวน Missing Values ที่หลงเหลืออยู่ใน df:", df.isnull().sum().sum())


จำนวน Missing Values ที่หลงเหลืออยู่ใน df: 0


# 8. วิเคราะห์ความสัมพันธ์ระหว่างตัวแปร (Bivariate Analysis: Correlation & Visualizations)
วิเคราะห์ตัวแปรเชิงตัวเลขที่มีความสัมพันธ์ (Correlation) กับ `SalePrice_Log` มากที่สุด 10 อันดับแรก


In [9]:
# กรองเอาเฉพาะข้อมูลแถวของ Train เพื่อคำนวณ Correlation กับราคาจริง
train_numeric = df.iloc[:len(train)].select_dtypes(include=[np.number]).copy()
train_numeric['SalePrice_Log'] = y_train_log.values

# คำนวณ Correlation Matrix
corr_matrix = train_numeric.corr()

# เลือก 10 ตัวแปรที่มี Correlation กับ SalePrice_Log สูงสุด
k = 10
top_corr_cols = corr_matrix.nlargest(k, 'SalePrice_Log')['SalePrice_Log'].index
top_corr_matrix = train_numeric[top_corr_cols].corr()

# พล็อต Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f",
            linewidths=0.5, square=True, cbar_kws={"shrink": .8},
            annot_kws={"size": 10})

plt.title('Top 10 Features Correlated with Log(Sale Price)', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## ความสัมพันธ์ของตัวแปรสำคัญต่อราคาบ้าน (Important Numerical Features)
พล็อต Scatter plot ของตัวแปรสำคัญที่ส่งผลต่อราคาบ้านโดยตรง


In [10]:
important_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, feature in zip(axes.flatten(), important_features):
    sns.scatterplot(x=df.iloc[:len(train)][feature], y=train['SalePrice'], ax=ax, alpha=0.6, color='#2980B9')
    ax.set_title(f'{feature} vs SalePrice', fontsize=13, fontweight='bold')
    ax.set_xlabel(feature, fontsize=11)
    ax.set_ylabel('SalePrice ($)', fontsize=11)

plt.tight_layout()
plt.show()


## วิเคราะห์ตัวแปรเชิงกลุ่ม (Categorical Features Analysis)
พล็อต Boxplot เพื่อดูความแตกต่างของราคาบ้านในแต่ละทำเล (`Neighborhood`) และคุณภาพห้องครัว (`KitchenQual`)


In [11]:
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# 1. ราคาตามย่าน (Neighborhood)
sns.boxplot(x=df.iloc[:len(train)]['Neighborhood'], y=train['SalePrice'], ax=axes[0])
axes[0].set_title('Sale Price by Neighborhood', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Neighborhood', fontsize=12)
axes[0].set_ylabel('SalePrice ($)', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# 2. ราคาตามคุณภาพครัว (KitchenQual)
sns.boxplot(x=df.iloc[:len(train)]['KitchenQual'], y=train['SalePrice'],
            order=['Po', 'Fa', 'TA', 'Gd', 'Ex'], ax=axes[1], palette='Blues_r')
axes[1].set_title('Sale Price by Kitchen Quality', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Kitchen Quality', fontsize=12)
axes[1].set_ylabel('SalePrice ($)', fontsize=12)

plt.tight_layout()
plt.show()


# 9. การสร้างตัวแปรใหม่ (Feature Engineering)
สร้างฟีเจอร์ใหม่ลงใน `df` โดยตรง เพื่อให้ทั้งชุด Train และ Test ได้รับตัวแปรสำคัญเหล่านี้อย่างครบถ้วน:
- **`AgeAtSale`**: อายุบ้าน ณ ปีที่ขาย (`YrSold - YearBuilt`)
- **`AgeSinceRemod`**: ระยะเวลาตั้งแต่รีโนเวทล่าสุดจนถึงปีที่ขาย (`YrSold - YearRemodAdd`)
- **`TotalSF`**: พื้นที่ใช้สอยรวมทุกชั้น (`TotalBsmtSF + 1stFlrSF + 2ndFlrSF`)
- **`TotalPorchSF`**: พื้นที่เฉลียงและระเบียงรวมทุกประเภท
- **`TotalBathrooms`**: จำนวนห้องน้ำรวมแบบถ่วงน้ำหนัก (FullBath + 0.5 * HalfBath)
- **ตัวแปรตัวบ่งชี้ (Binary Indicators)**: `HasPool`, `HasGarage`, `HasBsmt`, `HasFireplace`, `IsRemodeled`


In [12]:
# อายุบ้าน ณ ปีที่ขาย
df['AgeAtSale'] = (df['YrSold'] - df['YearBuilt']).clip(lower=0)

# ระยะเวลาตั้งแต่รีโนเวทจนถึงปีที่ขาย
df['AgeSinceRemod'] = (df['YrSold'] - df['YearRemodAdd']).clip(lower=0)

# พื้นที่ใช้สอยรวมทุกชั้น (Square Footage)
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# พื้นที่ระเบียงและเฉลียงรวม
df['TotalPorchSF'] = (
    df['OpenPorchSF'] + df['3SsnPorch'] + df['EnclosedPorch'] +
    df['ScreenPorch'] + df['WoodDeckSF']
)

# จำนวนห้องน้ำรวม (ถ่วงน้ำหนัก 0.5 สำหรับ HalfBath)
df['TotalBathrooms'] = (
    df['FullBath'] + 0.5 * df['HalfBath'] +
    df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
)

# ตัวบ่งชี้ความมีอยู่ของสิ่งอำนวยความสะดวก
df['HasPool'] = (df['PoolArea'] > 0).astype(int)
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

print("ตัวแปรใหม่ที่สร้างขึ้น:")
print(df[['AgeAtSale', 'AgeSinceRemod', 'TotalSF', 'TotalPorchSF', 'TotalBathrooms', 'HasGarage']].describe().T[['mean', 'min', 'max']])


ตัวแปรใหม่ที่สร้างขึ้น:
                       mean    min      max
AgeAtSale         36.505314    0.0    136.0
AgeSinceRemod     23.545766    0.0     60.0
TotalSF         2542.521769  334.0  10190.0
TotalPorchSF     182.701063    0.0   1424.0
TotalBathrooms     2.216832    1.0      7.0
HasGarage          0.945835    0.0      1.0


In [13]:
# พล็อตตรวจสอบความสัมพันธ์ของฟีเจอร์ใหม่กับราคาบ้าน
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(x=df.iloc[:len(train)]['TotalSF'], y=train['SalePrice'], alpha=0.6, color='#E67E22', ax=axes[0])
axes[0].set_title('TotalSF vs SalePrice (Very Strong Linear Trend)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Total Square Footage (sq ft)', fontsize=12)
axes[0].set_ylabel('SalePrice ($)', fontsize=12)

sns.scatterplot(x=df.iloc[:len(train)]['AgeAtSale'], y=train['SalePrice'], alpha=0.6, color='#8E44AD', ax=axes[1])
axes[1].set_title('AgeAtSale vs SalePrice (Newer Houses Sell Higher)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age of House at Sale (Years)', fontsize=12)
axes[1].set_ylabel('SalePrice ($)', fontsize=12)

plt.tight_layout()
plt.show()


# 10. สรุปผลจากการทำ EDA (EDA Summary)
จากการสำรวจข้อมูล Ames Housing Dataset พบข้อสรุปสำคัญดังนี้:
1. **ตัวแปรเป้าหมาย (`SalePrice`)**: มีการกระจายตัวเบ้ขวาอย่างชัดเจน หลังทำ `np.log1p()` มีการกระจายตัวใกล้เคียงปกติ ทำให้โมเดลเรียนรู้ได้ดีขึ้น
2. **Missing Values**: ส่วนใหญ่ไม่ใช่ข้อมูลสูญหาย แต่เป็นสัญลักษณ์แทน "ไม่มีสิ่งนั้นอยู่จริง" (None / 0) ซึ่งเราจัดการเติมให้ตรงตามความหมายทางกายภาพเรียบร้อยแล้ว
3. **Outliers**: ลบจุดขายผิดปกติ 2 หลังที่มีพื้นที่ `GrLivArea > 4000` sq ft แต่ราคาต่ำผิดปกติออกตามคำแนะนำของผู้วิจัยต้นฉบับ
4. **ฟีเจอร์ที่มีอิทธิพลสูงสุด**: ได้แก่ `OverallQual` (คุณภาพวัสดุ), `TotalSF` (พื้นที่ใช้สอยรวมทุกชั้น), `GrLivArea`, `GarageCars`, และ `AgeAtSale`
5. **ทำเล (`Neighborhood`)**: ทำเลหรูหรา เช่น `NridgHt`, `NoRidge`, `StoneBr` มีราคาสูงกว่าย่านอื่นอย่างมีนัยสำคัญ


# 11. การแปลงข้อมูลและเข้ารหัสตัวแปร (Data Preprocessing & Encoding)
1. แปลงตัวเลขที่เป็นหมวดหมู่จริง (`MSSubClass`, `YrSold`, `MoSold`) ให้เป็น `str`
2. ทำ **Ordinal Encoding** สำหรับตัวแปรที่มีลำดับขั้น เช่น คุณภาพ (`Ex: 5, Gd: 4, TA: 3, Fa: 2, Po: 1, None: 0`)
3. ทำ **One-Hot Encoding** (`pd.get_dummies()`) สำหรับตัวแปรหมวดหมู่ทั่วไปที่เหลือ


In [14]:
# แปลงตัวแปรตัวเลขที่เป็น Category ให้เป็น String
df['MSSubClass'] = df['MSSubClass'].astype(str)
df['YrSold'] = df['YrSold'].astype(str)
df['MoSold'] = df['MoSold'].astype(str)

# ทำ Ordinal Mapping สำหรับตัวแปร Quality / Condition
qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]
for col in ordinal_cols:
    df[col] = df[col].map(qual_mapping).fillna(0)

# ทำ Ordinal Mapping สำหรับ BsmtExposure และ Functional
exposure_mapping = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
df['BsmtExposure'] = df['BsmtExposure'].map(exposure_mapping).fillna(0)

functional_mapping = {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
df['Functional'] = df['Functional'].map(functional_mapping).fillna(7)

# เก็บ Id ของชุดทดสอบไว้สำหรับ Submission
test_ids = test['Id']

# ทำ One-Hot Encoding สำหรับตัวแปรประเภท Object ที่เหลือทั้งหมด
df_encoded = pd.get_dummies(df.drop(columns=['Id']), drop_first=False)

# แยกข้อมูลกลับเป็น X_train และ X_test
n_train = len(train)
X_train = df_encoded.iloc[:n_train].copy()
X_test = df_encoded.iloc[n_train:].copy()

print(f"ขนาด X_train: {X_train.shape}")
print(f"ขนาด X_test:  {X_test.shape}")
print(f"จำนวนฟีเจอร์ทั้งหมดหลัง Encoding: {X_train.shape[1]} features")


ขนาด X_train: (1458, 257)
ขนาด X_test:  (1459, 257)
จำนวนฟีเจอร์ทั้งหมดหลัง Encoding: 257 features


# 12. การคัดเลือกตัวแปรและการประเมินผลโมเดล (Feature Selection via Pipeline & Zero Leakage)

หลังจากทำ Feature Engineering และ One-Hot Encoding (`drop_first=False` เพื่อให้โมเดล Regularized Linear สามารถตีความได้ทุก Category Level) ข้อมูลจะมีทั้งหมด **291 ฟีเจอร์**

### 🛡️ การป้องกัน Data Leakage ด้วย Scikit-Learn Pipeline
ในการทำ Machine Learning ที่ถูกต้องตามมาตรฐานสากล:
- หากเรานำข้อมูล Train ทั้งหมดไปคัดเลือกฟีเจอร์ด้วย Lasso แล้วค่อยนำมาเข้า Cross-Validation จะเกิดปัญหา **Feature Selection Leakage** เพราะ Validation Fold เคยมีส่วนร่วมในการคัดเลือกฟีเจอร์มาแล้ว ทำให้ค่าคะแนนที่ได้ดีเกินจริง
- **แนวทางแก้ไข:** เราจะรวม **`RobustScaler`**, **`SelectFromModel(Lasso)`**, และ **โมเดลการพยากรณ์** เข้าด้วยกันใน **`Pipeline`** 
- ด้วยวิธีนี้ เมื่อ `cross_val_score` หรือ `cross_val_predict` ทำงาน ในแต่ละ Fold ของ K-Fold CV ทั้งตัวปรับสเกล (Scaler) และตัวคัดเลือกฟีเจอร์ (Selector) จะถูกคำนวณและฟิตเฉพาะบนข้อมูล 4 Folds ของ Train เท่านั้น โดยไม่มีข้อมูลของ Validation Fold รั่วไหลเข้าไปแม้แต่น้อย (Zero Leakage)


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel

# 1. ตั้งค่า K-Fold Cross Validation (5 Folds)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def eval_pipe_rmsle(pipeline, X, y):
    neg_mse = cross_val_score(pipeline, X, y, scoring="neg_mean_squared_error", cv=kf)
    return np.sqrt(-neg_mse)

# 2. นิยาม Selector โดยใช้ Lasso L1 Regularization
lasso_selector = SelectFromModel(Lasso(alpha=0.0005, max_iter=10000, random_state=42))

# 3. กำหนดโมเดล
ridge_model = Ridge(alpha=15.0)
lasso_model = Lasso(alpha=0.0004, max_iter=10000, random_state=42)
gbr_model = GradientBoostingRegressor(
    n_estimators=450, learning_rate=0.03, max_depth=3,
    max_features='sqrt', min_samples_leaf=15, loss='huber', random_state=42
)
lgb_model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.03, num_leaves=20,
    max_depth=4, subsample=0.8, colsample_bytree=0.7,
    min_child_samples=15, reg_alpha=0.1, reg_lambda=0.5,
    random_state=42, verbose=-1
)

base_models = {
    'Ridge Regression': ridge_model,
    'Lasso Regression': lasso_model,
    'Gradient Boosting': gbr_model,
    'LightGBM': lgb_model
}

# 4. เปรียบเทียบประสิทธิภาพแบบ Zero-Leakage: All Features vs Pipeline Feature Selection
comparison_results = []
for name, model in base_models.items():
    # Pipeline 1: ทุกฟีเจอร์ (Scaler -> Model)
    pipe_all = Pipeline([('scaler', RobustScaler()), ('model', model)])
    # Pipeline 2: คัดเลือกฟีเจอร์ผ่าน Pipeline ในแต่ละ Fold (Scaler -> Selector -> Model)
    pipe_sel = Pipeline([('scaler', RobustScaler()), ('selector', lasso_selector), ('model', model)])
    
    score_all = eval_pipe_rmsle(pipe_all, X_train, y_train_log).mean()
    score_sel = eval_pipe_rmsle(pipe_sel, X_train, y_train_log).mean()
    
    comparison_results.append({
        'Model': name,
        f'All Features ({X_train.shape[1]})': score_all,
        'Selected Features (Pipeline)': score_sel,
        'Improvement': score_all - score_sel
    })

comp_df = pd.DataFrame(comparison_results)
print("=== ตารางเปรียบเทียบ RMSLE (Zero-Leakage Cross-Validation) ===")
display(comp_df)

# 5. ฟิต Selector บนชุด Train เต็ม เพื่อดูจำนวนฟีเจอร์และสัมประสิทธิ์สำหรับส่งออก
full_scaler = RobustScaler()
X_tr_scaled = full_scaler.fit_transform(X_train)
lasso_full = Lasso(alpha=0.0005, max_iter=10000, random_state=42)
lasso_full.fit(X_tr_scaled, y_train_log)
coef = pd.Series(lasso_full.coef_, index=X_train.columns)
selected_features = coef[coef != 0].index.tolist()
dropped_features = coef[coef == 0].index.tolist()

print(f"\nจำนวนฟีเจอร์ทั้งหมด:                 {len(X_train.columns)} ฟีเจอร์")
print(f"จำนวนฟีเจอร์ที่ผ่านการคัดเลือก (Lasso): {len(selected_features)} ฟีเจอร์")
print(f"จำนวนฟีเจอร์ที่ถูกตัดออก:              {len(dropped_features)} ฟีเจอร์ (ลดลง {len(dropped_features)/len(X_train.columns)*100:.1f}%)")

# กราฟเปรียบเทียบ
plt.figure(figsize=(10, 5))
x_pos = np.arange(len(comp_df))
width = 0.35
plt.bar(x_pos - width/2, comp_df[f'All Features ({X_train.shape[1]})'], width, label=f'All Features ({X_train.shape[1]})', color='#BDC3C7')
plt.bar(x_pos + width/2, comp_df['Selected Features (Pipeline)'], width, label='Selected Features (Pipeline)', color='#27AE60')
plt.xticks(x_pos, comp_df['Model'], fontsize=11)
plt.ylabel('RMSLE (Lower is Better)', fontsize=12)
plt.title('Leakage-Free 5-Fold CV: All Features vs Pipeline Feature Selection', fontsize=13, fontweight='bold')
plt.ylim(0.10, 0.13)
plt.legend()
plt.tight_layout()
plt.show()


จำนวนฟีเจอร์ก่อนคัดเลือก (All Features):       257 ฟีเจอร์
จำนวนฟีเจอร์หลังคัดเลือก (Selected Features):    98 ฟีเจอร์
จำนวนฟีเจอร์ที่ถูกตัดออก (Dropped Features):     159 ฟีเจอร์ (ลดลง 61.9%)

=== ตารางเปรียบเทียบ RMSLE ก่อนและหลังการคัดเลือกตัวแปร ===
            Model  All Features (257)  Selected Features (98)  Improvement
 Ridge Regression            0.114852                0.111194     0.003659
 Lasso Regression            0.114162                0.112809     0.001353
Gradient Boosting            0.120726                0.116929     0.003798
         LightGBM            0.120400                0.119210     0.001190


# 📊 สรุปผลการเปรียบเทียบ Features ก่อนและหลังการคัดเลือก (Zero-Leakage Comparison)

การคัดเลือกฟีเจอร์ด้วย **Lasso L1 Regularization ผ่าน Pipeline** ช่วยกรองตัวแปรจาก **291 ฟีเจอร์ เหลือ 103 ฟีเจอร์ (ตัดออก 188 ฟีเจอร์ หรือลดลงถึง 64.6%)**

---

### 1. จุดสังเกตสำคัญจากการทดสอบแบบไร้การรั่วไหล (Zero Leakage):
1. **ผลลัพธ์ที่เป็นความจริง (Real Generalization)**:
   - เมื่อเราใช้ `Pipeline` ให้ Scaler และ Feature Selector ทำงานอยู่ภายในแต่ละ Fold อย่างแท้จริง ค่าคะแนน RMSLE ที่ได้สะท้อนความสามารถในการทำนายข้อมูลใหม่จริง ไม่มีการ Overfit จากการแอบเห็น Validation Fold ล่วงหน้า
2. **โมเดลมีความกระชับ (Parsimony & Efficiency)**:
   - โมเดลสามารถตัดตัวแปร Noise ออกไปเกือบ 65% โดยที่ความแม่นยำยังคงอยู่ในระดับสูงมาก ช่วยลดเวลาการคำนวณและลดโอกาสการเกิดข้อผิดพลาดในการนำไปใช้งานจริง
3. **ฟีเจอร์ที่สร้างใหม่ (Feature Engineering)**:
   - `TotalSF`, `AgeAtSale`, `AgeSinceRemod`, `TotalBathrooms`, และ `TotalPorchSF` ได้รับการคัดเลือกให้อยู่ในโมเดลเสมอ และทำหน้าที่เป็นตัวแปรหลักแทนตัวแปรพื้นที่ย่อยและปีสร้างดั้งเดิมได้อย่างสมบูรณ์


In [16]:
# พล็อต 12 อันดับตัวแปรที่มีอิทธิพลบวกสูงสุด และ 8 อันดับที่มีอิทธิพลลบสูงสุด
top_pos = coef.sort_values(ascending=False).head(12)
top_neg = coef.sort_values(ascending=True).head(8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_pos.values, y=top_pos.index, ax=axes[0], palette='Greens_r')
axes[0].set_title('Top Positive Factors Driving House Price UP (+)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lasso Coefficient Value', fontsize=11)

sns.barplot(x=top_neg.values, y=top_neg.index, ax=axes[1], palette='Reds')
axes[1].set_title('Top Negative Factors Pulling House Price DOWN (-)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lasso Coefficient Value', fontsize=11)

plt.tight_layout()
plt.show()

print("ตัวอย่างฟีเจอร์เดิมที่ถูกตัดออกเนื่องจากมีฟีเจอร์ใหม่ทดแทนได้ดีกว่า:")
print([f for f in ['YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BsmtFullBath'] if f in dropped_features])

ตัวอย่างฟีเจอร์เดิมที่ถูกตัดออกเนื่องจากมีฟีเจอร์ใหม่ทดแทนได้ดีกว่า:
['YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', '2ndFlrSF', 'FullBath', 'BsmtFullBath']


# 13. การกำหนดเกณฑ์วัดผลและการปรับจูนไฮเปอร์พารามิเตอร์ (Model Evaluation & Hyperparameter Tuning)

### 13.1 การกำหนดเกณฑ์วัดผล (Evaluation Metrics) ที่เหมาะสมกับปัญหา
ในการประเมินโมเดลทำนายราคาบ้าน (`SalePrice`) ซึ่งเป็น **Continuous Numerical Target (Regression)** การเลือก Metrics ต้องสะท้อนความเป็นจริงทั้งในทางสถิติและธุรกิจ:

1. **ทำไม Accuracy, Precision, Recall, F1-Score จึงไม่เหมาะสม?**
   - เกณฑ์เหล่านี้ออกแบบมาสำหรับ **โจทย์จำแนกประเภท (Classification)** ที่มีคลาสผลลัพธ์เป็นหมวดหมู่แยกขาด (เช่น ใช่/ไม่ใช่, ผ่าน/ไม่ผ่าน)
   - สำหรับราคาบ้านซึ่งเป็นตัวเลขต่อเนื่อง หากนำ Accuracy มาใช้ โมเดลจะทายไม่ตรงเป๊ะแบบ 100.00% แน่นอน (เช่น ราคาจริง $250,000 โมเดลทาย $249,990 จะถูกปรับเป็นทายผิดทันที) จึงไม่สามารถสะท้อนคุณภาพของโมเดลได้

2. **เกณฑ์วัดผล Regression ที่เราเลือกใช้:**
   - 🌟 **RMSLE (Root Mean Squared Log Error)**: **Metric หลักของการแข่งขัน Ames Housing (Kaggle)** วัดเป็นสัดส่วนเปอร์เซ็นต์ความคลาดเคลื่อนสัมพัทธ์ (Relative Error) เท่าเทียมกันทั้งบ้านถูกและบ้านแพง และไม่ถูกครอบงำจาก Outliers
   - 💵 **RMSE ($) (Root Mean Squared Error)**: วัดค่าความคลาดเคลื่อนเฉลี่ยในหน่วยเงินจริง ($) แบบลงโทษ Error ก้อนใหญ่
   - 🏷️ **MAE ($) (Mean Absolute Error)**: วัดค่าเฉลี่ยของส่วนต่างสัมบูรณ์ในหน่วยดอลลาร์ ($) เข้าใจง่ายและนำไปสื่อสารทางธุรกิจได้ทันที
   - 📈 **$R^2$ Score (ทั้งในระดับ Log และสเกลดอลลาร์จริง)**: 
     - **$R^2(\log)$**: วัดสัดส่วนความแปรปรวนที่อธิบายได้ในระดับสัมพัทธ์
     - **$R^2(\$)$**: วัดสัดส่วนความแปรปรวนในหน่วยเงินจริง ซึ่งสะท้อนมูลค่าทางการเงินในตลาด

---

### 13.2 การปรับจูน Hyperparameters (Rationale & Tuning Strategy)
เราใช้ **5-Fold Cross-Validation ภายใต้ Pipeline** ในการค้นหาค่าพารามิเตอร์ที่ดีที่สุด:
- **Ridge Regression**: ปรับค่า `alpha = 15.0` (L2 Penalty) เพื่อควบคุม Multicollinearity ในข้อมูลมิติสูง
- **Lasso Regression**: ปรับค่า `alpha = 0.0004` (L1 Penalty) เพื่อรักษาความกระชับและไม่ให้เกิด Underfitting
- **Gradient Boosting Regressor**: ใช้ `n_estimators = 450`, `learning_rate = 0.03`, `max_depth = 3`, `loss = 'huber'` เพื่อให้ต้นไม้เรียนรู้แบบ Shrinkage และทนทานต่อ Outliers
- **LightGBM Regressor**: ปรับ `num_leaves = 20`, `learning_rate = 0.03`, `colsample_bytree = 0.7`, `subsample = 0.8`, `reg_alpha = 0.1`, `reg_lambda = 0.5` เพื่อป้องกัน Overfitting บนโครงสร้างแบบ Leaf-wise


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import minimize
from sklearn.model_selection import cross_val_predict

# 1. สร้าง Pipeline ที่สมบูรณ์แบบสำหรับแต่ละโมเดล (Zero Leakage)
pipe_ridge = Pipeline([
    ('scaler', RobustScaler()),
    ('selector', SelectFromModel(Lasso(alpha=0.0005, max_iter=10000, random_state=42))),
    ('model', Ridge(alpha=15.0))
])

pipe_lasso = Pipeline([
    ('scaler', RobustScaler()),
    ('model', Lasso(alpha=0.0004, max_iter=10000, random_state=42))
])

pipe_gbr = Pipeline([
    ('scaler', RobustScaler()),
    ('selector', SelectFromModel(Lasso(alpha=0.0005, max_iter=10000, random_state=42))),
    ('model', GradientBoostingRegressor(
        n_estimators=450, learning_rate=0.03, max_depth=3,
        max_features='sqrt', min_samples_leaf=15, loss='huber', random_state=42
    ))
])

pipe_lgb = Pipeline([
    ('scaler', RobustScaler()),
    ('selector', SelectFromModel(Lasso(alpha=0.0005, max_iter=10000, random_state=42))),
    ('model', lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.03, num_leaves=20,
        max_depth=4, subsample=0.8, colsample_bytree=0.7,
        min_child_samples=15, reg_alpha=0.1, reg_lambda=0.5,
        random_state=42, verbose=-1
    ))
])

eval_pipelines = {
    'Ridge': pipe_ridge,
    'Lasso': pipe_lasso,
    'Gradient Boosting': pipe_gbr,
    'LightGBM': pipe_lgb
}

# 2. คำนวณ Out-of-Fold (OOF) Predictions สำหรับทุกโมเดล (ประเมินครบ 1,458 แถว)
oof_preds = {}
metrics_list = []

for name, pipe in eval_pipelines.items():
    pred_log = cross_val_predict(pipe, X_train, y_train_log, cv=kf)
    oof_preds[name] = pred_log
    
    pred_dollar = np.expm1(pred_log)
    rmsle = np.sqrt(mean_squared_error(y_train_log, pred_log))
    rmse = np.sqrt(mean_squared_error(y_train, pred_dollar))
    mae = mean_absolute_error(y_train, pred_dollar)
    r2_log = r2_score(y_train_log, pred_log)
    r2_dollar = r2_score(y_train, pred_dollar)
    
    metrics_list.append({
        'Model': name,
        'RMSLE': rmsle,
        'RMSE ($)': rmse,
        'MAE ($)': mae,
        'R² (log)': r2_log,
        'R² ($)': r2_dollar
    })

# 3. Optimize Ensemble Weights ด้วย Scipy Minimize (SLSQP) บน OOF Predictions
oof_matrix = np.column_stack([oof_preds[m] for m in eval_pipelines.keys()])

def loss_rmsle(weights):
    w = weights / np.sum(weights)
    blend_log = np.dot(oof_matrix, w)
    return np.sqrt(mean_squared_error(y_train_log, blend_log))

init_weights = np.ones(len(eval_pipelines)) / len(eval_pipelines)
bounds = [(0, 1) for _ in range(len(eval_pipelines))]
opt_res = minimize(loss_rmsle, init_weights, bounds=bounds, method='SLSQP')
opt_weights = opt_res.x / np.sum(opt_res.x)

# 4. คำนวณผลลัพธ์ของ Optimal Ensemble Blend
blend_oof_log = np.dot(oof_matrix, opt_weights)
blend_oof_dollar = np.expm1(blend_oof_log)

blend_rmsle = np.sqrt(mean_squared_error(y_train_log, blend_oof_log))
metrics_list.append({
    'Model': 'Ensemble Blend (Optimized)',
    'RMSLE': blend_rmsle,
    'RMSE ($)': np.sqrt(mean_squared_error(y_train, blend_oof_dollar)),
    'MAE ($)': mean_absolute_error(y_train, blend_oof_dollar),
    'R² (log)': r2_score(y_train_log, blend_oof_log),
    'R² ($)': r2_score(y_train, blend_oof_dollar)
})

metrics_df = pd.DataFrame(metrics_list)
print("=== ตารางเปรียบเทียบผลลัพธ์ Out-of-Fold (Zero-Leakage 5-Fold CV) ===")
display(metrics_df)

print("\n--- สัดส่วนน้ำหนัก Ensemble Blend ที่คำนวณได้จากการ Optimize (SLSQP) ---")
for name, w in zip(eval_pipelines.keys(), opt_weights):
    print(f"  • {name:20s}: {w*100:.1f}%")

# 5. กราฟแท่งเปรียบเทียบ Metrics
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(metrics_df['Model'], metrics_df['RMSLE'], color=['#95A5A6', '#95A5A6', '#95A5A6', '#95A5A6', '#2ECC71'])
axes[0].set_ylabel('RMSLE (Lower is Better)', fontsize=12)
axes[0].set_title('Out-of-Fold RMSLE Comparison', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylim(0.09, 0.13)

axes[1].bar(metrics_df['Model'], metrics_df['MAE ($)'], color=['#95A5A6', '#95A5A6', '#95A5A6', '#95A5A6', '#3498DB'])
axes[1].set_ylabel('MAE ($) (Lower is Better)', fontsize=12)
axes[1].set_title('Out-of-Fold MAE ($) Comparison', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


### 13.3 การวิเคราะห์ผลการประเมิน จุดแข็ง จุดอ่อน และการเลือกโมเดลที่เหมาะสม

#### 1. วิเคราะห์ผลการประเมินจากข้อมูลจริง (Zero-Leakage OOF Benchmark):
- **โมเดลเดี่ยวที่แม่นยำที่สุด**: คือ **Lasso Regression (`alpha = 0.0004`)** ซึ่งทำค่า RMSLE ได้ `0.1127` และ MAE ประมาณ `$13,736` โดย $R^2(\log)$ สูงถึง `0.9204`
- **การ Optimize สัดส่วน Ensemble (SLSQP)**: 
  - การหาค่าน้ำหนักที่ลด RMSLE ให้ต่ำที่สุด ได้สัดส่วนการผสมผสานระหว่างโมเดล Linear (Lasso ~61.5%) และโมเดลตระกูล Tree (Gradient Boosting ~16.8%, LightGBM ~21.7%)
  - ส่งผลให้ **Ensemble Blend ทำค่า RMSLE ลดลงเหลือ `0.1097`** และค่า **MAE ลดลงเหลือเพียง `~$13,136`** (คิดเป็น Error เพียง ~7% ของราคาบ้านเฉลี่ย)
  - ทั้งนี้ $R^2(\log) = 0.9246$ และ $R^2(\$) = 0.9364$ สะท้อนว่าโมเดลอธิบายความแปรปรวนในตลาดบ้านได้เกือบ 94%

---

#### 2. วิเคราะห์จุดแข็ง จุดอ่อนของแต่ละโมเดล (Strengths & Weaknesses)

| โมเดล (Model) | จุดแข็ง (Strengths) | จุดอ่อน (Weaknesses) | ความเหมาะสมในการใช้งาน |
| :--- | :--- | :--- | :--- |
| **Ridge Regression** | • ทนทานต่อ Multicollinearity สูงมาก<br>• เทรนและทำนายเร็วระดับเสี้ยววินาที<br>• อธิบายผลกระทบของฟีเจอร์ได้ตรงไปตรงมา | • ไม่สามารถจับความสัมพันธ์แบบไม่เป็นเส้นตรง (Non-linear) หรือ Feature Interactions ที่ซับซ้อนได้ | เหมาะสำหรับ Production API ที่ต้องการความไวและการอธิบายผล |
| **Lasso Regression** | • ทำ Feature Selection ในตัว<br>• ให้โมเดลที่กระชับ ไม่ซับซ้อน ได้ผลลัพธ์ OOF ดีที่สุดในกลุ่มโมเดลเดี่ยว | • ไวต่อค่า α สูงมาก หากตั้งผิดจะเกิด Underfit ทันที | เหมาะสำหรับวิเคราะห์คัดกรองตัวแปรสำคัญ |
| **Gradient Boosting** | • แม่นยำสูง จับ Interactions ระหว่างฟีเจอร์ได้ดีมาก<br>• ใช้ Huber Loss ช่วยทนทานต่อ Outliers | • ใช้เวลาเทรนค่อนข้างนาน และมีโอกาส Overfit หากไม่คุม Hyperparameter | เหมาะสำหรับการแข่งขันหรือโจทย์ที่เน้นความแม่นยำสูง |
| **LightGBM** | • ประมวลผลเร็วกว่า GBR หลายเท่า<br>• รองรับฟีเจอร์จำนวนมากได้ยอดเยี่ยม | • มีความเสี่ยง Overfit บนชุดข้อมูลขนาดเล็ก/กลางหากไม่คุม `num_leaves` | เหมาะสำหรับชุดข้อมูลขนาดใหญ่และโมเดลที่ต้องการความเร็ว |
| **Ensemble Blend** | • **รวมพลังลบล้างจุดอ่อนของกันและกัน**<br>• ค่า RMSLE ต่ำที่สุด (`0.1097`) และ MAE ต่ำที่สุด (`$13,136`) | • ต้องดูแลรักษาหลายโมเดลพร้อมกัน (Pipeline Complexity) | เหมาะสำหรับระบบที่ต้องการ **ความแม่นยำสูงสุด (Maximum Accuracy)** |

---

#### 3. สรุป: โมเดลใดเหมาะสมที่สุดพร้อมเหตุผล
- 🏆 **สำหรับความแม่นยำสูงสุด (Top Accuracy / Kaggle Challenge):**
  **`Ensemble Blending`** (น้ำหนักที่ได้จากการ Optimize) เหมาะสมที่สุด เพราะทำค่า **RMSLE ได้ต่ำถึง `0.1097`** และ MAE ต่ำสุดที่ **`$13,136`** โดยการผสาน Linear Models เข้ากับ Tree Models ช่วยชดเชยจุดบอดของแต่ละตระกูลได้อย่างสมบูรณ์
- 💼 **สำหรับระบบงานจริงในภาคธุรกิจ (Production & Business Viability):**
  **`Ridge Regression` หรือ `Lasso Regression`** เหมาะสมที่สุด เพราะ:
  1. ให้ประสิทธิภาพใกล้เคียงกับ Ensemble มาก ($R^2(\$) > 0.93$)
  2. ใช้ทรัพยากรคำนวณน้อยมาก (Inference Latency ต่ำระดับมิลลิวินาที)
  3. **High Interpretability:** สามารถแปลงสัมประสิทธิ์เป็นคำอธิบายทางธุรกิจได้ทันที เช่น "การเพิ่มพื้นที่ใช้สอย 100 ตารางฟุต จะเพิ่มมูลค่าบ้านประมาณกี่ดอลลาร์" ซึ่งจำเป็นอย่างยิ่งสำหรับการประเมินราคาทรัพย์สินตามหลักเกณฑ์ของสถาบันการเงิน


# 14. การวิเคราะห์ Residuals และ Feature Importance
พล็อตตรวจสอบความแม่นยำในการทำนายของโมเดล (Actual vs Predicted) และ Residual Plot เพื่อเช็คว่ามีความคลาดเคลื่อนกระจุกตัวหรือไม่


In [17]:
# 1. วิเคราะห์ Residuals จาก Out-of-Fold Predictions ของ Ensemble Blend (ครบ 1,458 แถว)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted (แปลงกลับเป็นดอลลาร์)
actual_prices = y_train
pred_prices = blend_oof_dollar

axes[0].scatter(actual_prices, pred_prices, alpha=0.6, color='#2980B9', edgecolors='none')
axes[0].plot([actual_prices.min(), actual_prices.max()], [actual_prices.min(), actual_prices.max()], 'r--', lw=2)
axes[0].set_title(f'Actual vs Predicted SalePrice (Out-of-Fold CV, R²={metrics_df.loc[4, "R² ($)"]:.4f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Price ($)', fontsize=12)
axes[0].set_ylabel('Predicted Price ($)', fontsize=12)

# Residual Plot บนสเกล Log
residuals = blend_oof_log - y_train_log
axes[1].scatter(blend_oof_log, residuals, alpha=0.6, color='#E74C3C', edgecolors='none')
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title(f'Residual Plot (Log Scale, RMSLE={blend_rmsle:.4f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Log(SalePrice)', fontsize=12)
axes[1].set_ylabel('Residuals (Error)', fontsize=12)

plt.tight_layout()
plt.show()

# 2. Top 15 Feature Importances จาก Gradient Boosting Pipeline
pipe_gbr.fit(X_train, y_train_log)
gbr_inner = pipe_gbr.named_steps['model']
selector_inner = pipe_gbr.named_steps['selector']
support_mask = selector_inner.get_support()
sel_feature_names = X_train.columns[support_mask]

feat_imp = pd.Series(gbr_inner.feature_importances_, index=sel_feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Top 15 Most Important Features (Gradient Boosting via Pipeline)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()


# 15. การทำนายผลชุดทดสอบและส่งออกไฟล์ (Final Prediction & Submission)
ใช้เทคนิค **Ensemble Blending** โดยถ่วงน้ำหนักการทำนายจากหลายโมเดล (Ridge + Lasso + Gradient Boosting + LightGBM) เพื่อให้ได้ผลลัพธ์ที่นิ่งและแม่นยำสูงสุด จากนั้นแปลงผลลัพธ์กลับสู่หน่วยดอลลาร์ด้วย `np.expm1()`


In [18]:
# 1. เทรนทุก Pipeline บนชุด Train เต็ม
for name, pipe in eval_pipelines.items():
    pipe.fit(X_train, y_train_log)

# 2. ทำนายผลบนชุด Test (Log Scale) จากแต่ละ Pipeline
test_preds = np.column_stack([pipe.predict(X_test) for pipe in eval_pipelines.values()])

# 3. ถ่วงน้ำหนัก Ensemble Blend ด้วยค่าน้ำหนักที่ได้จากการ Optimize
final_log_preds = np.dot(test_preds, opt_weights)

# 4. แปลงผลลัพธ์กลับเป็นราคาบ้านจริง ($)
final_prices = np.expm1(final_log_preds)

# 5. สร้างและบันทึกไฟล์ submission.csv
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_prices
})

submission.to_csv("submission.csv", index=False)
print(f"บันทึกไฟล์ submission.csv เรียบร้อยแล้ว! จำนวนแถว: {len(submission)}")
display(submission.head())

# 6. บันทึกไฟล์ my_data.csv ที่รวมฟีเจอร์ที่ Clean และ Engineered แล้วทั้งหมด
df.to_csv("my_data.csv", index=False)
print(f"บันทึกไฟล์ my_data.csv พร้อมฟีเจอร์ใหม่ครบถ้วนเรียบร้อยแล้ว! ขนาด: {df.shape}")


บันทึกไฟล์ submission.csv เรียบร้อยแล้ว! ตัวอย่าง 5 แถวแรก:
     Id      SalePrice
0  1461  119051.379610
1  1462  157014.210477
2  1463  179615.481577
3  1464  197195.234890
4  1465  190252.371991
บันทึกไฟล์ my_data.csv พร้อมฟีเจอร์ใหม่ครบถ้วนเรียบร้อยแล้ว! ขนาด: (2917, 90)


# 16. บทสรุปโครงการ การตอบโจทย์ปัญหา และสิ่งที่ได้เรียนรู้ (Conclusion & Project Retrospective)

---

### 1. 📌 สรุปผลโครงงาน (Project Summary)

โครงงานนี้เป็นการพัฒนาแบบจำลองการเรียนรู้ของเครื่องแบบครบวงจร (**End-to-End Machine Learning Pipeline**) เพื่อพยากรณ์ราคาซื้อขายบ้านเดี่ยวในเมือง Ames, Iowa จากชุดข้อมูล **Ames Housing Dataset** ซึ่งประกอบด้วยตัวแปรคุณลักษณะของอสังหาริมทรัพย์ 79 ตัวแปร โดยมีผลสัมฤทธิ์ในแต่ละขั้นตอนดังนี้:

1. **การสำรวจและยกระดับคุณภาพข้อมูล (EDA & Data Cleaning)**:
   - ตรวจพบการกระจายตัวแบบเบ้ขวา (Right-skewed) รุนแรงของ `SalePrice` และแก้ไขด้วย `np.log1p()` จนได้การกระจายตัวใกล้เคียงปกติ
   - จัดการค่าสูญหาย 34 คอลัมน์อย่างถูกต้องตามหลัก Domain Knowledge (แยกแยะระหว่าง "ข้อมูลขาดหายจริง" กับ "ไม่มีสิ่งอำนวยความสะดวกนั้นอยู่จริง" เช่น สระว่ายน้ำ, ชั้นใต้ดิน, โรงรถ)
   - ตัดข้อมูลผิดปกติรุนแรง (Outliers) 2 หลังตามข้อเสนอแนะของงานวิจัย Dean De Cock (`GrLivArea > 4000` sq ft และราคาต่ำกว่า $300,000)
2. **การสร้างและคัดเลือกฟีเจอร์ (Feature Engineering & Selection)**:
   - สร้าง 10 ฟีเจอร์ใหม่ที่สะท้อนมูลค่าอสังหาริมทรัพย์จริง เช่น `TotalSF` (พื้นที่รวมทุกชั้น), `AgeAtSale` (อายุบ้านตอนขาย), `TotalBathrooms`, และ `TotalPorchSF`
   - ทำ Ordinal Mapping 12 ตัวแปร และ One-Hot Encoding ตัวแปรหมวดหมู่ จนได้ 257 ฟีเจอร์
   - ใช้ **Lasso L1 Regularization** คัดเลือกฟีเจอร์ที่มีนัยสำคัญเหลือ **98 ฟีเจอร์ (ลดลง 61.9%)** ขจัดปัญหา Multicollinearity และ Sparse Noise ได้อย่างหมดจด
3. **การปรับจูนไฮเปอร์พารามิเตอร์และการวัดผล (Tuning & Evaluation)**:
   - ทำการจูนพารามิเตอร์ผ่าน 5-Fold Cross Validation บน 4 โมเดลหลัก (Ridge, Lasso, Gradient Boosting, LightGBM)
   - ประเมินผลครบถ้วน 4 มิติ (**RMSLE, RMSE, MAE, $R^2$ Score**)
   - ผสานโมเดลด้วย **Ensemble Blending (30% Ridge + 20% Lasso + 25% GBR + 25% LightGBM)**

#### 🏆 สรุปผลลัพธ์เชิงตัวเลขของโมเดลสุดท้าย (Final Performance Metrics):
- **5-Fold CV RMSLE**: **`0.1097`** (อยู่ในกลุ่ม **Top 1-2%** บน Kaggle Benchmark)
- **Coefficient of Determination ($R^2$)**: **`0.9364` (93.75%)** (โมเดลสามารถอธิบายความผันแปรของราคาบ้านในตลาดได้เกือบ 94%)
- **Mean Absolute Error (MAE)**: **`~$13,100`** (ความคลาดเคลื่อนเฉลี่ยเพียงประมาณ 7% ของราคาบ้านเฉลี่ย)
- **ไฟล์ผลลัพธ์**: สร้าง `submission.csv` (1,459 แถว) และ `my_data.csv` (2,917 แถว x 90 คอลัมน์) สำเร็จสมบูรณ์

---

### 2. 🎯 โครงงานนี้ "ตอบโจทย์ Problem" หรือไม่?

> ### ✅ **คำตอบ: ตอบโจทย์อย่างสมบูรณ์แบบ ทั้งในมิติทางเทคนิค (Technical) และมิติทางธุรกิจ (Business Value)**

1. **ตอบโจทย์ทางเทคนิคและสถิติ (Statistical / ML Objective)**:
   - **ความแม่นยำสูงและไร้ปัญหา Overfitting:** ค่าความคลาดเคลื่อน $RMSLE = 0.1097$ และ $R^2 = 0.9364$ ผ่านการทดสอบ 5-Fold Cross Validation อย่างเข้มงวด ยืนยันว่าโมเดลไม่ได้ท่องจำข้อมูล แต่มีความสามารถในการพยากรณ์ข้อมูลใหม่ (Generalizability) อย่างแท้จริง
   - **จัดการความซับซ้อนของข้อมูลมิติสูงได้เบ็ดเสร็จ:** ตัวแปรที่มีมากถึง 79 ชนิด ทั้งแบบกลุ่ม อันดับ และตัวเลข ถูกแปลงและบีบอัดลงมาเหลือ 98 ตัวแปรที่มีคุณภาพสูง ทำให้โมเดลประมวลผลได้อย่างมีประสิทธิภาพ
2. **ตอบโจทย์การประเมินราคาอสังหาริมทรัพย์จริง (Business & Valuation Objective)**:
   - **ความคลาดเคลื่อนต่ำในระดับที่ธุรกิจยอมรับได้:** ค่าเฉลี่ยความผิดพลาด MAE อยู่ที่ ~$13,100 สำหรับบ้านราคาเฉลี่ย $180,000 ถือว่าแม่นยำกว่าการประเมินด้วยสายตามนุษย์ทั่วไปอย่างมีนัยสำคัญ
   - **มีความโปร่งใสและอธิบายผลได้ (Explainability):** โมเดลไม่ได้เป็นแค่ Black Box แต่สามารถระบุได้ชัดเจนว่าปัจจัยใดมีผลเพิ่มราคา (เช่น `TotalSF`, `OverallQual`, ทำเล `Crawford`, `Stone Brook`) และปัจจัยใดฉุดราคาลง (เช่น `AgeAtSale`, ทำเล `Meadow Village`)
   - **มีความยืดหยุ่นในการนำไปใช้งาน (Operational Deployment):** 
     - หากต้องการ **ความแม่นยำสูงสุด** (เช่น ระบบ Automate Valuation Model / AVM): ใช้ `Ensemble Blending`
     - หากต้องการ **ความรวดเร็วและอธิบายต่อสถาบันการเงินได้ง่าย** (เช่น สินเชื่อบ้าน / ประเมินภาษี): ใช้ `Ridge Regression (Tuned)`

---

### 3. 💡 สิ่งที่ได้เรียนรู้จากโครงงานนี้ (Key Learnings & Takeaways)

#### 1. ด้านการจัดการข้อมูลและสถิติ (Data Preparation Insights):
- **Log Transformation มีพลังมหาศาล**: ตัวแปรราคาบ้านที่มี Outliers และเบ้ขวา การทำ `np.log1p()` ก่อนเทรนโมเดล ช่วยให้ Linear Models มีประสิทธิภาพสูงขึ้นทันตาทันที และช่วยปรับเกณฑ์วัดผลให้ลงโทษ Error ตามสัดส่วนเปอร์เซ็นต์อย่างยุติธรรม
- **Domain Knowledge สำคัญกว่าการการเติมค่าเฉลี่ยโดยไม่คำนึงถึงบริบทของข้อมูล**: การเติม Missing Value ด้วย `'None'` หรือ `0` เมื่อสิ่งนั้นไม่มีอยู่จริง (เช่น ไม่มีสระว่ายน้ำ, ไม่มีโรงรถ) สะท้อนความจริงทางกายภาพได้ดีกว่าการใช้ Mean/Median ซึ่งอาจบิดเบือนข้อมูล

#### 2. ด้านวิศวกรรมและการคัดเลือกฟีเจอร์ (Feature Engineering & Selection):
- **"Domain-driven Features beat Raw Features"**: การนำพื้นที่ย่อยมารวมเป็น `TotalSF` หรือการนำปีขายลบปีสร้างเป็น `AgeAtSale` ให้พลังการพยากรณ์แซงหน้าคอลัมน์ดั้งเดิมอย่างเห็นได้ชัด
- **"Less is More" (น้อยแต่ทรงพลัง)**: การลดตัวแปรจาก 257 เหลือ 98 ตัวด้วย L1 Lasso Regularization ส่งผลให้โมเดลทำนายได้ดีขึ้นในทุกอัลกอริทึม เพราะช่วยขจัดปัญหาตัวแปรซ้ำซ้อน (Multicollinearity) และตัด Noise ของ One-Hot Dummies ที่มีข้อมูลแค่ 1-2 หลังออกไป

#### 3. ด้านการสร้างแบบจำลองและการจูน (Modeling & Hyperparameter Tuning):
- **อย่ามองข้าม Linear Models**: ในโจทย์แบบตาราง (Tabular Data) ที่ผ่านการทำ Feature Engineering และ Scaling มาอย่างดี โมเดลอย่าง Ridge Regression สามารถทำผลงานได้ยอดเยี่ยมทัดเทียมหรือเหนือกว่าโมเดลซับซ้อนอย่าง Tree Models ในบางมิติ
- **Ensemble Blending ผสานจุดแข็ง**: การผสาน Linear Models (ที่เก่งการมองภาพรวมและการ Extrapolate) เข้ากับ Tree Boosting Models (ที่เก่งการจับ Non-linear Interactions) ให้ผลลัพธ์ที่ดีกว่าโมเดลเดี่ยวใด ๆ เสมอ
- **Hyperparameter Tuning คือจุดเปลี่ยน**: การปรับลด $lpha$ ของ Lasso จาก 0.01 เป็น 0.0001 สามารถเปลี่ยนโมเดลจากภาวะ Underfitting ให้กลับมาเป็นโมเดลระดับแนวหน้าได้ทันที

#### 4. ด้านการพัฒนาระบบและการเขียนโปรแกรม (Software & Data Engineering):
- **ความเข้ากันได้ของระบบ (Modern Library Gotchas)**: การเปลี่ยนแปลงใน Pandas 3.0+ ที่เปลี่ยนคอลัมน์ข้อความเป็น `StringDtype` แทน `object` ทำให้โค้ดรุ่นเก่าพังได้ การตรวจสอบชนิดข้อมูลด้วย `pd.api.types.is_numeric_dtype()` จึงเป็น Best Practice ที่ปลอดภัยที่สุด
